# Split de datos para PlantVillage

Este notebook realiza el split estratificado del dataset de PlantVillage en conjuntos de entrenamiento, validación y prueba, siguiendo los requerimientos del proyecto. Los splits se guardan en archivos JSON para asegurar la reproducibilidad.

## 1. Importar librerías necesarias
Importamos las librerías para manipulación de datos y el split estratificado.

In [ ]:
import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

## 2. Cargar y explorar los datos

Cargamos la información de las imágenes y sus etiquetas. Asumimos que las imágenes están organizadas en carpetas por clase dentro de `data/processed/`.

In [ ]:
# Escanear carpetas y crear un DataFrame con rutas y etiquetas
image_dir = '../data/processed/'
classes = [d for d in os.listdir(image_dir) if os.path.isdir(os.path.join(image_dir, d))]

image_paths = []
labels = []
for cls in classes:
    cls_dir = os.path.join(image_dir, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(cls_dir, fname))
            labels.append(cls)

df = pd.DataFrame({'image_path': image_paths, 'label': labels})
df.head()

## 3. Definir los requerimientos del proyecto para el split

- Split estratificado por clase
- Proporciones: 70% entrenamiento, 15% validación, 15% prueba
- Guardar los splits en archivos JSON para reproducibilidad

## 4. Realizar el split de los datos según los requerimientos

Se realiza el split estratificado y se guardan los resultados en archivos JSON.

In [ ]:
# Split estratificado
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

# Guardar splits en JSON
os.makedirs('../data/splits', exist_ok=True)
train_df[['image_path', 'label']].to_json('../data/splits/train.json', orient='records', lines=True)
val_df[['image_path', 'label']].to_json('../data/splits/val.json', orient='records', lines=True)
test_df[['image_path', 'label']].to_json('../data/splits/test.json', orient='records', lines=True)

## 5. Verificar y mostrar los resultados del split

Mostramos el número de ejemplos por clase en cada subconjunto y algunos ejemplos para validar que el split es correcto.

In [ ]:
print('Distribución por clase en train:')
print(train_df['label'].value_counts())
print('\nDistribución por clase en val:')
print(val_df['label'].value_counts())
print('\nDistribución por clase en test:')
print(test_df['label'].value_counts())

print('\nEjemplo de train:')
print(train_df.sample(3))
print('\nEjemplo de val:')
print(val_df.sample(3))
print('\nEjemplo de test:')
print(test_df.sample(3))